In [1]:
import os
import json
import ast
import re
import gc
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from tqdm import tqdm

import torch
import torch.nn.functional as F

import faiss 
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

/home/rayane/Desktop/Paris Cité/M1/PPD/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# LLM (generator)
LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Dense retriever (embeddings)
RETRIEVER_NAME = "Qwen/Qwen3-Embedding-0.6B"

DATA_TRAIN = "data/train.csv"
DATA_VALID = "data/validation.csv"

SEED = 42

# Retrieval / indexing
TOP_K = 5
CHUNK_WORDS = 200
CHUNK_OVERLAP = 50
RET_MAX_TOKENS = 512
RET_BATCH_SIZE = 64
QUERY_BATCH_SIZE = 64

# Generation
MAX_INPUT_TOKENS = 768
MAX_NEW_TOKENS = 32
GEN_BATCH_SIZE = 1  



# Cache 
CACHE_DIR = "results/rag_cache"
INDEX_PATH = os.path.join(CACHE_DIR, f"faiss_ip_dim1024_cw{CHUNK_WORDS}_ov{CHUNK_OVERLAP}.index")
CHUNKS_PATH = os.path.join(CACHE_DIR, f"chunks_cw{CHUNK_WORDS}_ov{CHUNK_OVERLAP}.jsonl")

# Outputs
METRICS_CSV = "results/metrics_table_qwen7B_rag.csv"
PRED200_CSV = "results/preds200_qwen7B_rag.csv"

DEVICE = "cuda" 

In [3]:
def parse_answers_field(x):
    s = str(x).strip()
    s2 = re.sub(r"array\(\s*(\[[\s\S]*?\])\s*,\s*dtype=[^)]+\)", r"\1", s)
    ans = ast.literal_eval(s2)
    ans["text"] = list(ans["text"])
    ans["answer_start"] = [int(v) for v in ans["answer_start"]]
    return ans

def normalize_dataset(example):
    example["answers"] = parse_answers_field(example["answers"])
    example["id"] = str(example["id"])
    return example

def build_prompt(question, tokenizer, context = None):
    question = str(question).strip()

    system_msg = (
        "You are a helpful assistant. "
        "Answer the question with a short phrase. "
        "Do not add explanations. "
        "If the context does not contain the answer, say: I don't know."
    )

    if context is not None and len(context.strip()) > 0:
        user_msg = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    else:
        user_msg = f"Question: {question}\nAnswer:"

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return f"{system_msg}\n\n{user_msg}"

    
def clean_generation(text):
    t = (text or "").strip()
    t = t.split("\n")[0].strip()
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    # keep it short-ish
    return t.strip()

In [4]:
def chunk_text_words(text, chunk_words, overlap):
    text = (text or "").strip()
    if not text:
        return []
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    chunks = []
    step = max(1, chunk_words - overlap)
    for start in range(0, len(words), step):
        end = start + chunk_words
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(words):
            break
    return chunks

In [5]:
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom

@torch.inference_mode()
def embed_texts(
    texts,
    tokenizer,
    model,
    max_tokens = 512,
    batch_size = 32,
    prefix = None,
    device = "cuda"):
    
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        if prefix is not None:
            batch = [prefix + t for t in batch]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_tokens,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        pooled = F.normalize(pooled, p=2, dim=1)
        vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
        del enc, out, pooled
    return np.vstack(vecs) if vecs else np.zeros((0, 1024), dtype=np.float32)


In [6]:
def save_chunks_jsonl(path, chunks):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for t in chunks:
            f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")

def load_chunks_jsonl(path):
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            chunks.append(obj["text"])
    return chunks


def build_or_load_faiss_index(
    ds_train,
    retr_tok,
    retr_model,
    dim = 1024):
    os.makedirs(CACHE_DIR, exist_ok=True)

    if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
        index = faiss.read_index(INDEX_PATH)
        chunks = load_chunks_jsonl(CHUNKS_PATH)
        return index, chunks

    # Build corpus chunks from TRAIN contexts only
    raw_contexts = ds_train["context"]
    seen = set()
    corpus_chunks = []

    for ctx in raw_contexts:
        ctx = (ctx or "").strip()
        if not ctx:
            continue
        # optional dedup at full-context level
        if ctx in seen:
            continue
        seen.add(ctx)
        parts = chunk_text_words(ctx, CHUNK_WORDS, CHUNK_OVERLAP)
        corpus_chunks.extend(parts)

    # Create FAISS (cosine via inner product on normalized vectors)
    index = faiss.IndexFlatIP(dim)

    # Add in batches
    for i in tqdm(range(0, len(corpus_chunks), RET_BATCH_SIZE), desc="Indexing corpus (embeddings + FAISS)"):
        batch = corpus_chunks[i:i+RET_BATCH_SIZE]
        emb = embed_texts(
            batch,
            tokenizer=retr_tok,
            model=retr_model,
            max_tokens=RET_MAX_TOKENS,
            batch_size=min(RET_BATCH_SIZE, 64),
            prefix="passage: ",
            device=DEVICE,
        )
        if emb.shape[0] > 0:
            index.add(emb)

    faiss.write_index(index, INDEX_PATH)
    save_chunks_jsonl(CHUNKS_PATH, corpus_chunks)
    return index, corpus_chunks

In [7]:
def retrieve_topk_contexts(
    questions,
    retr_tok,
    retr_model,
    index,
    chunks,
    top_k = 5):
    retrieved_contexts = []
    for i in tqdm(range(0, len(questions), QUERY_BATCH_SIZE), desc="Retrieving top-k contexts"):
        batch_q = questions[i:i+QUERY_BATCH_SIZE]
        q_emb = embed_texts(
            batch_q,
            tokenizer=retr_tok,
            model=retr_model,
            max_tokens=RET_MAX_TOKENS,
            batch_size=min(QUERY_BATCH_SIZE, 64),
            prefix="query: ",
            device=DEVICE,
        )
        D, I = index.search(q_emb, top_k)
        for row in I:
            texts = [chunks[int(idx)] for idx in row if int(idx) >= 0 and int(idx) < len(chunks)]
            # Simple concat; you can change separator later
            retrieved_contexts.append("\n\n".join(texts))
    return retrieved_contexts

In [8]:
def load_llm_qwen_7b():
    llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, use_fast=True)
    llm_tok.padding_side = "left"
    llm_tok.truncation_side = "left"
    if llm_tok.pad_token_id is None:
        llm_tok.pad_token = llm_tok.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
    )
    llm.eval()
    llm.config.use_cache = False
    return llm_tok, llm

In [9]:
set_seed(SEED)
os.makedirs("results", exist_ok=True)

# Load data
ds = load_dataset("csv", data_files={"train": DATA_TRAIN, "validation": DATA_VALID})
ds = ds.map(normalize_dataset)

train_split = ds["train"]
val_split = ds["validation"]

Generating train split: 87599 examples [00:00, 166405.72 examples/s]
Generating validation split: 1000 examples [00:00, 83845.83 examples/s]
Map: 100%|████████████████████████| 1000/1000 [00:00<00:00, 27698.70 examples/s]


In [10]:
# Load retriever (embeddings model)
retr_tok = AutoTokenizer.from_pretrained(RETRIEVER_NAME, use_fast=True)
retr_model = AutoModel.from_pretrained(
    RETRIEVER_NAME,
    torch_dtype=torch.float16,
)
retr_model.eval()
retr_model.to(DEVICE)

`torch_dtype` is deprecated! Use `dtype` instead!


Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [11]:
# Build or load FAISS index
index, corpus_chunks = build_or_load_faiss_index(train_split, retr_tok, retr_model, dim=1024)

# Retrieve contexts for validation questions
val_questions = list(val_split["question"])
retrieved_ctxs = retrieve_topk_contexts(val_questions, retr_tok, retr_model, index, corpus_chunks, top_k=TOP_K)


def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

hits = 0
total = len(val_split)
for i in range(total):
    gold = val_split[i]["answers"]["text"][0] if val_split[i]["answers"]["text"] else ""
    if gold and normalize_text(gold) in normalize_text(retrieved_ctxs[i]):
        hits += 1

retrieval_hit_rate = 100.0 * hits / max(1, total)
print(f"Retrieval hit rate (gold in retrieved context): {retrieval_hit_rate:.2f}%")

del retr_model
del retr_tok
torch.cuda.empty_cache()
gc.collect()

Retrieving top-k contexts: 100%|████████████████| 16/16 [00:02<00:00,  5.51it/s]


Retrieval hit rate (gold in retrieved context): 15.10%


742

In [12]:
# Load LLM (if OOM, free retriever and retry)
try:
    llm_tok, llm = load_llm_qwen_7b()
except torch.cuda.OutOfMemoryError:
    if "retr_model" in locals():
        del retr_model
    if "retr_tok" in locals():
        del retr_tok
    torch.cuda.empty_cache()
    gc.collect()
    llm_tok, llm = load_llm_qwen_7b()


Loading checkpoint shards: 100%|██████████████████| 4/4 [00:13<00:00,  3.44s/it]


In [ ]:
# Generation on validation with retrieved contexts
squad_metric = evaluate.load("squad")
predictions = []
rows_200 = []

ids = list(val_split["id"])
gold_answers = [ex["answers"]["text"][0] if ex["answers"]["text"] else "" for ex in val_split]

for start in tqdm(range(0, len(val_split), GEN_BATCH_SIZE), desc="Generating (RAG, Qwen 7B)"):
    batch_ids = ids[start:start + GEN_BATCH_SIZE]
    batch_q = val_questions[start:start + GEN_BATCH_SIZE]
    batch_ctx = retrieved_ctxs[start:start + GEN_BATCH_SIZE]

    prompts = [build_prompt(q, llm_tok, context=c) for q, c in zip(batch_q, batch_ctx)]
    inputs = llm_tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    # move inputs to the first cuda device if available
    if DEVICE == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.inference_mode():
        out = llm.generate(
            **inputs,
            use_cache=False,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            eos_token_id=llm_tok.eos_token_id,
            pad_token_id=llm_tok.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[:, prompt_len:]
    gen_texts = llm_tok.batch_decode(gen_ids, skip_special_tokens=True)

    del inputs, out, gen_ids
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    for ex_id, gt in zip(batch_ids, gen_texts):
        pred = clean_generation(gt)
        predictions.append({"id": str(ex_id), "prediction_text": pred})

Generating (RAG, Qwen 7B):   0%|             | 1/1000 [00:18<5:13:16, 18.82s/it]

In [ ]:
references = [{"id": str(ex["id"]), "answers": ex["answers"]} for ex in val_split]
results = squad_metric.compute(predictions=predictions, references=references)

metrics_df = pd.DataFrame([{
    "split": "validation",
    "exact_match": results["exact_match"],
    "f1": results["f1"],
    "model": LLM_NAME,
    "retriever": RETRIEVER_NAME,
    "top_k": TOP_K,
    "chunk_words": CHUNK_WORDS,
    "chunk_overlap": CHUNK_OVERLAP,
    "retrieval_hit_rate": retrieval_hit_rate,
}])

print("\n=== RAG (Qwen 7B) Validation Results ===")
print(metrics_df.to_string(index=False))

In [ ]:
metrics_df.to_csv(METRICS_CSV, index=False)

# Build 200-first CSV: id, question, gold, generated answer
pred_by_id = {p["id"]: p["prediction_text"] for p in predictions}
pred200 = []
for i in range(min(200, len(val_split))):
    ex_id = str(ids[i])
    pred200.append({
        "id": ex_id,
        "question": val_questions[i],
        "gold_answer": gold_answers[i],
        "generated_answer": pred_by_id.get(ex_id, ""),
    })
pred200_df = pd.DataFrame(pred200)
pred200_df.to_csv(PRED200_CSV, index=False)

print("\nSaved:")
print(" -", METRICS_CSV)
print(" -", PRED200_CSV)
